In [ ]:
pip install xarray
pip install netCDF4
pip install netcdf4 h5netcdf


In [ ]:
file_path = r"file_path"


from netCDF4 import Dataset
ds = Dataset(file_path, mode='r')

print(ds)


In [ ]:
import xarray as xr

file_path = r"file_path"

ds = xr.open_dataset(file_path, engine="netcdf4")
print(ds)


In [ ]:
ds['t2m_c'] = ds['t2m'] - 273.15

df = ds[['t2m_c']].to_dataframe().reset_index()
df.head()


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))
plt.plot(df['valid_time'], df['t2m_c'])
plt.title("Temperature Time Series")
plt.xlabel("Time")
plt.ylabel("Temperature (°C)")
plt.show()


In [ ]:
df['target'] = df['t2m_c'].shift(-24)
df = df.dropna()

for i in range(1,25):
    df[f'temp_t-{i}'] = df['t2m_c'].shift(i)

df = df.dropna()


In [ ]:
split = int(len(df)*0.8)

train = df[:split]
test = df[split:]

X_train = train[[f'temp_t-{i}' for i in range(1,25)]]
y_train = train['target']

X_test = test[[f'temp_t-{i}' for i in range(1,25)]]
y_test = test['target']


In [ ]:
baseline_pred = X_test['temp_t-24']


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

mae = mean_absolute_error(y_test, baseline_pred)
rmse = np.sqrt(mean_squared_error(y_test, baseline_pred))

print("Baseline MAE:", mae)
print("Baseline RMSE:", rmse)


In [ ]:
#linear regressio
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

pred = model.predict(X_test)

from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))

print("Linear Regression MAE:", mae)
print("Linear Regression RMSE:", rmse)


In [ ]:
#random forest
from sklearn.ensemble importRandomForestRegressor

rf = RandomForestRegressor(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

mae_rf = mean_absolute_error(y_test, rf_pred)
rmse_rf = np.sqrt(mean_squared_error(y_test, rf_pred))

print("Random Forest MAE:", mae_rf)
print("Random Forest RMSE:", rmse_rf)


In [ ]:
#xg boost 
pip install xgboost

from xgboost import XGBRegressor

xgb = XGBRegressor(n_estimators=200, learning_rate=0.05)
xgb.fit(X_train, y_train)

xgb_pred = xgb.predict(X_test)

mae_xgb = mean_absolute_error(y_test, xgb_pred)
rmse_xgb = np.sqrt(mean_squared_error(y_test, xgb_pred))

print("XGBoost MAE:", mae_xgb)
print("XGBoost RMSE:", rmse_xgb)

In [ ]:

from sklearn.svm import SVR

svr = SVR(kernel='rbf')
svr.fit(X_train, y_train)

svr_pred = svr.predict(X_test)

mae_svr = mean_absolute_error(y_test, svr_pred)
rmse_svr = np.sqrt(mean_squared_error(y_test, svr_pred))

print("SVR MAE:", mae_svr)
print("SVR RMSE:", rmse_svr)

In [ ]:
#Gradient Boost
from sklearn.ensemble import GradientBoostingRegressor

gbr = GradientBoostingRegressor()
gbr.fit(X_train, y_train)

gbr_pred = gbr.predict(X_test)

mae_gbr = mean_absolute_error(y_test, gbr_pred)
rmse_gbr = np.sqrt(mean_squared_error(y_test, gbr_pred))

print("GBR MAE:", mae_gbr)
print("GBR RMSE:", rmse_gbr)

In [ ]:
import matplotlib.pyplot as plt

models = ["Baseline","Linear","RF","GBR","SVR","XGBoost"]
mae = [1.88,1.42,1.42,1.46,1.49,1.51] ## recived values for above train

plt.bar(models, mae)
plt.ylabel("MAE (°C)")
plt.title("Model Benchmark Comparison")
plt.show()

In [ ]:
#coampre with linear , random forest and actual one
import matplotlib.pyplot as plt

plt.figure(figsize=(12,6))
plt.plot(y_test.values, label="Actual")
plt.plot(pred, label="Linear Regression")
plt.plot(rf_pred, label="Random Forest")
plt.legend()
plt.title("24-Hour Ahead Temperature Forecast")
plt.show()


In [ ]:
# for one week data 

In [ ]:
import xarray as xr

test_file = r"file_path"

ds_test = xr.open_dataset(test_file)
print(ds_test)

In [ ]:
ds_test["t2m_c"] = ds_test["t2m"] - 273.15

df_test = ds_test[["t2m_c"]].to_dataframe().reset_index()

df_test.head()

In [ ]:
for i in range(1,25):
    df_test[f"temp_t-{i}"] = df_test["t2m_c"].shift(i)

df_test = df_test.dropna()

In [ ]:
X_test_new = df_test[[f"temp_t-{i}" for i in range(1,25)]]
y_test_new = df_test["t2m_c"]

In [ ]:
lr_pred = model.predict(X_test_new)

from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

mae_lr = mean_absolute_error(y_test_new, lr_pred)
rmse_lr = np.sqrt(mean_squared_error(y_test_new, lr_pred))

print("LR Jan2026 MAE:", mae_lr)
print("LR Jan2026 RMSE:", rmse_lr)

In [ ]:
# all 5 models
rf_pred = rf.predict(X_test_new)

mae_rf = mean_absolute_error(y_test_new, rf_pred)
rmse_rf = np.sqrt(mean_squared_error(y_test_new, rf_pred))

xgb_pred = xgb.predict(X_test_new)

mae_xgb = mean_absolute_error(y_test_new, xgb_pred)
rmse_xgb = np.sqrt(mean_squared_error(y_test_new, xgb_pred))

svr_pred = svr.predict(X_test_new)

gbr_pred = gbr.predict(X_test_new)

In [ ]:
#actual vs linear
import matplotlib.pyplot as plt

plt.figure(figsize=(12,6))
plt.plot(y_test_new.values, label="Actual")
plt.plot(lr_pred, label="Linear Regression")
plt.legend()
plt.title("Temperature Forecast – Jan 2026 Test Week")
plt.show()

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

def evaluate_model(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    print(f"{name} MAE:", mae)
    print(f"{name} RMSE:", rmse)
    print("----------------------")

In [ ]:
lr_pred = model.predict(X_test_new)
evaluate_model("Linear Regression", y_test_new, lr_pred)


rf_pred = rf.predict(X_test_new)
evaluate_model("Random Forest", y_test_new, rf_pred)


gbr_pred = gbr.predict(X_test_new)
evaluate_model("Gradient Boosting", y_test_new, gbr_pred)


xgb_pred = xgb.predict(X_test_new)
evaluate_model("XGBoost", y_test_new, xgb_pred)


svr_pred = svr.predict(X_test_new)
evaluate_model("SVR", y_test_new, svr_pred)